# Caduceus on Apple Silicon — research analysis

Profiles the MPS port of Caduceus across:

1. Setup & device
2. Load pretrained Ph + PS
3. Parameter breakdown (with weight-tying detection)
4. Inputs & encoding (tokenization → embedding lookup → RC-equivariant embedding for PS)
5. Forward outputs (logits, softmax heatmap, top-k decoded predictions, per-position entropy)
6. **Masked-LM accuracy** (top-1 / top-3, cross-entropy, perplexity, per-base confusion matrix)
7. Performance: wall-clock vs sequence length
8. Memory profile vs sequence length
9. Mamba dynamics: effective memory spectrum
10. dt and A_log structure across layers
11. Activation sparsity & dead channels
12. RC equivariance verification on Caduceus-PS
13. Suggested follow-ups

Run from the repo root: `jupyter notebook notebooks/caduceus_analysis.ipynb`. Models download on first run (~30 MB each).

## 1. Setup

In [ ]:
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Make the repo importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from caduceus.modeling_caduceus import CaduceusForMaskedLM, BiMambaWrapper
from caduceus.tokenization_caduceus import CaduceusTokenizer
from caduceus.mamba_pytorch import Mamba, Block, RMSNorm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch {torch.__version__}  device={device}")

## 2. Load pretrained Caduceus-Ph and Caduceus-PS

In [ ]:
PH_ID = "kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16"
PS_ID = "kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16"

ph = CaduceusForMaskedLM.from_pretrained(PH_ID, fused_add_norm=False).eval().to(device)
ps = CaduceusForMaskedLM.from_pretrained(PS_ID, fused_add_norm=False).eval().to(device)
tok = CaduceusTokenizer(model_max_length=131072)
print("loaded")

## 3. Parameter breakdown

Caduceus has weight-tied parameters (`mamba_rev.in_proj`/`out_proj` share storage with `mamba_fwd` when `bidirectional_weight_tie=True`). PyTorch's `parameters()` iterator deduplicates by tensor identity, so `unique_params` reflects the real footprint. The breakdown below groups by component.

In [ ]:
def param_breakdown(model, name):
    by_kind = defaultdict(int)
    seen_ptrs = set()
    for full_name, p in model.named_parameters():
        # Bucket by the last component-name segment we care about
        if ".embeddings." in full_name:
            bucket = "embedding"
        elif ".lm_head" in full_name:
            bucket = "lm_head"
        elif "norm_f" in full_name:
            bucket = "final_norm"
        elif ".norm." in full_name or full_name.endswith(".norm.weight"):
            bucket = "per_layer_norm"
        elif "mamba_fwd" in full_name:
            bucket = "mamba_fwd"
        elif "mamba_rev" in full_name:
            bucket = "mamba_rev_unique"
        else:
            bucket = "other"
        # Only count each unique tensor once
        ptr = p.data_ptr()
        if ptr in seen_ptrs:
            continue
        seen_ptrs.add(ptr)
        by_kind[bucket] += p.numel()
    total = sum(by_kind.values())
    print(f"\n{name}: {total:,} unique params, {total*4/1e6:.1f} MB fp32")
    for k, v in sorted(by_kind.items(), key=lambda x: -x[1]):
        print(f"  {k:25s}  {v:>10,}  ({v/total*100:4.1f}%)")
    return by_kind

_ = param_breakdown(ph, "Caduceus-Ph")
_ = param_breakdown(ps, "Caduceus-PS")

## 4. Inputs & encoding

A DNA sequence reaches the model in two stages:

1. **Char → token id (tokenization).** 16-token vocab: 7 special tokens (`[CLS]`, `[SEP]`, `[BOS]`, `[MASK]`, `[PAD]`, `[RESERVED]`, `[UNK]`) + the five nucleotides A, C, G, T, N. The complement map sends A↔T (7↔10), C↔G (8↔9), N→N — used by Caduceus-PS for reverse-complement equivariance.
2. **Token id → embedding vector.** A learned lookup table of shape `(vocab_size, d_model)`. Caduceus-Ph uses a vanilla `nn.Embedding`. Caduceus-PS uses an `RCPSEmbedding` that constructs the embedding so `embed(rc(x))` is the **flipped reverse-complement** of `embed(x)` — RC equivariance is baked into the encoding step itself.

In [ ]:
# --- Stage 1: tokenization (char -> token id) ---
print("vocab (token -> id):")
for s, i in sorted(tok._vocab_str_to_int.items(), key=lambda x: x[1]):
    print(f"  {i:>3d}  {s!r}")
print("\ncomplement_map (Ph & PS share the same map):")
print(" ", {int(k): v for k, v in ph.config.complement_map.items()})

seq = "ACGTTACCGGAATTACGT" * 8  # 144 nt, deliberately not its own reverse complement
ids = torch.tensor([tok(seq)["input_ids"]], dtype=torch.long).to(device)

print(f"\ninput sequence (first 24 chars): {seq[:24]!r}")
print(f"tokenized shape: {tuple(ids.shape)}  (B=1, L includes a leading [CLS])")
print(f"first 24 token ids: {ids[0, :24].cpu().tolist()}")
print("\nchar  ->  id  (first 12 positions, after CLS):")
inv_vocab = {v: k for k, v in tok._vocab_str_to_int.items()}
for pos, tid in enumerate(ids[0, :13].cpu().tolist()):
    if pos == 0:
        print(f"  pos {pos:>3d}  [CLS]  -> {tid}")
    else:
        print(f"  pos {pos:>3d}    {seq[pos-1]}    -> {tid}  ({inv_vocab[tid]!r})")

In [ ]:
# --- Stage 2a: token id -> embedding vector (Caduceus-Ph, vanilla nn.Embedding) ---
emb_layer = ph.caduceus.backbone.embeddings.word_embeddings
print(f"Caduceus-Ph embedding table: {tuple(emb_layer.weight.shape)}  "
      f"(vocab_size={ph.config.vocab_size}, d_model={ph.config.d_model})")

with torch.no_grad():
    embeds_ph = emb_layer(ids)                        # (1, L, d_model)
print(f"\ninputs (ids):     shape={tuple(ids.shape)}      dtype={ids.dtype}")
print(f"embeddings (Ph):  shape={tuple(embeds_ph.shape)}  dtype={embeds_ph.dtype}")
print(f"  -> per-position vector norm range: "
      f"[{embeds_ph.norm(dim=-1).min().item():.3f}, {embeds_ph.norm(dim=-1).max().item():.3f}]")

# Show the embeddings of A/C/G/T as compact vectors, then the cosine similarity matrix
acgt_ids = [tok._vocab_str_to_int[c] for c in "ACGT"]
acgt_emb = emb_layer.weight[acgt_ids].detach().cpu()
sims = F.cosine_similarity(acgt_emb.unsqueeze(0), acgt_emb.unsqueeze(1), dim=-1).numpy()
print(f"\nA/C/G/T embedding cosine similarity (Caduceus-Ph):")
print("       " + "  ".join(f"{c:>6}" for c in "ACGT"))
for i, c in enumerate("ACGT"):
    print(f"  {c}  " + "  ".join(f"{sims[i, j]:>6.3f}" for j in range(4)))

In [ ]:
# --- Stage 2b: PS RC-equivariant encoding ---
# RCPSEmbedding builds a (B, L, 2*d_model) embedding such that
#   embed(rc(x)) = flip(embed(x), dims=[L, channel])
# That property propagates through every layer and is what gives Caduceus-PS
# its by-construction RC equivariance. Verify it numerically here.

cmap = {int(k): v for k, v in ps.config.complement_map.items()}
rc_ids = torch.tensor(
    [[cmap[t] for t in ids[0].tolist()][::-1]],
    dtype=torch.long, device=device,
)

ps_emb_layer = ps.caduceus.backbone.embeddings.word_embeddings
print(f"PS RCPSEmbedding internal table: {tuple(ps_emb_layer.embedding.weight.shape)}")
print(f"PS RCPSEmbedding output (per RCPS construction):  (B, L, 2 * d_model)")

with torch.no_grad():
    e_x = ps_emb_layer(ids)                           # (1, L, 2*d_model)
    e_rc = ps_emb_layer(rc_ids)
    # The RCPS property: embed(rc(x)) == flip(embed(x), dims=[L, channel])
    e_x_flipped = torch.flip(e_x, dims=[-2, -1])
    diff = (e_rc - e_x_flipped).abs().max().item()

print(f"\nembed(rc(x)) shape:  {tuple(e_rc.shape)}")
print(f"flip(embed(x))[L,C]: {tuple(e_x_flipped.shape)}")
print(f"max abs diff: {diff:.2e}   (architectural — must be ~0)")
assert diff < 1e-6, "RC-equivariant embedding property is broken"
print("=> RC-equivariance is enforced at the embedding step itself.")

## 5. Forward pass — outputs

The model returns logits over the 16-token vocab at every input position: shape `(B, L, vocab_size)`. We look at:

1. The shape, value range, and top-1 token recovery at unmasked positions.
2. A softmax heatmap to see where the model is confident vs. uniform.
3. Top-3 decoded predictions per position (with probabilities) for the first stretch of the sequence.
4. Per-position entropy — *where is the model uncertain?*

In [ ]:
with torch.no_grad():
    out_ph = ph(ids).logits
    out_ps = ps(ids).logits

for name, logits in [("Caduceus-Ph", out_ph), ("Caduceus-PS", out_ps)]:
    top1 = logits[0].argmax(dim=-1)
    matches = (top1[1:] == ids[0, 1:]).sum().item()
    print(f"{name}: shape={tuple(logits.shape)}  range=[{logits.min().item():.2f}, {logits.max().item():.2f}]  top-1 recovery {matches}/{ids.shape[1]-1}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, (name, logits) in zip(axes, [("Caduceus-Ph", out_ph), ("Caduceus-PS", out_ps)]):
    probs = F.softmax(logits[0], dim=-1).cpu().numpy()
    # Show the 11 "interesting" tokens (drop padding/reserved at the tail)
    im = ax.imshow(probs[:, :11].T, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_yticks(range(11))
    ax.set_yticklabels([f"{i}" for i in range(11)])
    ax.set_ylabel("token id")
    ax.set_title(f"{name}: softmax distribution per position")
axes[-1].set_xlabel("sequence position")
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label="prob")
plt.show()

In [ ]:
# Top-3 decoded predictions per position + per-position entropy.
inv_vocab_full = {v: k for k, v in tok._vocab_str_to_int.items()}
n_show = 16  # first n_show positions of the un-masked forward pass

print(f"{'pos':>3}  {'input':>5}  {'top-3 (token  prob)':<48}  {'entropy':>8}")
probs_ph = F.softmax(out_ph[0], dim=-1)
for pos in range(n_show):
    input_tok = inv_vocab_full[ids[0, pos].item()]
    top3 = probs_ph[pos].topk(3)
    top3_str = "  ".join(
        f"{inv_vocab_full[t.item()]:>5}({p.item():.2f})" for p, t in zip(top3.values, top3.indices)
    )
    H = -(probs_ph[pos] * (probs_ph[pos].clamp_min(1e-12)).log()).sum().item()
    print(f"{pos:>3}  {input_tok:>5}  {top3_str:<48}  {H:>8.3f}")

# Per-position entropy line (tells you where the model is confident vs. uncertain)
entropy_ph = -(probs_ph * probs_ph.clamp_min(1e-12).log()).sum(dim=-1).cpu().numpy()
probs_ps = F.softmax(out_ps[0], dim=-1)
entropy_ps = -(probs_ps * probs_ps.clamp_min(1e-12).log()).sum(dim=-1).cpu().numpy()
plt.figure(figsize=(12, 3))
plt.plot(entropy_ph, label="Caduceus-Ph", alpha=0.8)
plt.plot(entropy_ps, label="Caduceus-PS", alpha=0.8)
plt.axhline(np.log(16), color="gray", linestyle="--", label=f"uniform over 16 = {np.log(16):.2f}")
plt.axhline(np.log(4), color="orange", linestyle="--", label=f"uniform over A/C/G/T = {np.log(4):.2f}")
plt.xlabel("sequence position")
plt.ylabel("entropy (nats)")
plt.title("Per-position output entropy — lower = more confident")
plt.legend(loc="upper right", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Masked-LM accuracy

The training objective: replace some positions with `[MASK]` and predict the original tokens. We measure:

- **Top-1 / Top-3 accuracy** at masked positions only.
- **Cross-entropy in nats** (lower is better; uniform-over-4-nucleotides is `ln(4) ≈ 1.386`).
- **Perplexity** = `exp(CE)` (closer to 1 is better, 4.0 is uniform-over-nucleotides).
- **Per-base confusion matrix** — does the model confuse A↔T or C↔G specifically?

**Caveats** (don't over-interpret the absolute numbers):
- The eval uses a synthetic ACGT-repeat sequence with deterministic 8-mer periodicity. For Caduceus-Ph this is essentially a memorization task; for Caduceus-PS the RC-equivariance constraint changes how it generalizes from local context. Differences between Ph and PS here reflect inductive bias on synthetic input as much as model quality.
- For real biological accuracy you'd run the same code on held-out chromosome regions (e.g., the eval splits used in the Caduceus paper).

In [ ]:
# BERT-style masking: pick 15% of non-CLS positions, replace with [MASK], predict only at those.
# Caveat: `seq` is a synthetic ACGT-repeat with deterministic 8-mer periodicity, which makes
# this an *easy* eval for a competent DNA LM. For real biological accuracy you'd run this
# on a held-out chromosome region; the goal here is to demonstrate the methodology and
# verify the loaded weights produce sensible predictions on novel positions.
mask_token = tok._vocab_str_to_int["[MASK]"]
torch.manual_seed(0)

# Use a longer sequence for more masked-position samples; keep `ids` (144 nt) intact for later cells.
long_seq = ("ACGTTACCGGAATTACGT" * 32) + ("GCTAGCTAGCAATCGTACGT" * 16)  # ~896 nt
ids_eval = torch.tensor([tok(long_seq)["input_ids"]], dtype=torch.long).to(device)

mask_prob = 0.15
mask = torch.rand(ids_eval.shape, device=device) < mask_prob
mask[:, 0] = False  # never mask CLS
masked_ids = ids_eval.clone()
masked_ids[mask] = mask_token
n_masked = mask.sum().item()
print(f"sequence length: {ids_eval.shape[1]} tokens   masked positions: {n_masked} ({n_masked/ids_eval.shape[1]*100:.1f}%)\n")

# Uniform baseline: 4 nucleotide tokens out of 16 vocab tokens. If we knew the answer was a
# nucleotide we'd get 1/4 = 25% top-1, 3/4 = 75% top-3. Without that prior, uniform is 1/16.
print(f"{'model':>14}  {'top-1':>8}  {'top-3':>8}  {'CE (nats)':>10}  {'PPL':>7}")
print(f"{'uniform-4':>14}  {25.0:>7.2f}%  {75.0:>7.2f}%  {np.log(4):>10.3f}  {4.0:>7.2f}")
for name, model in [("Caduceus-Ph", ph), ("Caduceus-PS", ps)]:
    with torch.no_grad():
        logits = model(masked_ids).logits  # (1, L, V)
    masked_logits = logits[mask]           # (n_masked, V)
    masked_true = ids_eval[mask]           # (n_masked,)
    top1 = masked_logits.argmax(dim=-1)
    top3 = masked_logits.topk(3, dim=-1).indices
    top1_acc = (top1 == masked_true).float().mean().item()
    top3_acc = (top3 == masked_true.unsqueeze(-1)).any(dim=-1).float().mean().item()
    ce = F.cross_entropy(masked_logits, masked_true).item()
    print(f"{name:>14}  {top1_acc*100:>7.2f}%  {top3_acc*100:>7.2f}%  {ce:>10.3f}  {np.exp(ce):>7.2f}")

In [ ]:
# Per-base confusion: when masking an A vs C vs G vs T, what does the model predict?
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
nuc_ids = [tok._vocab_str_to_int[c] for c in "ACGT"]
nuc_idx = {tid: i for i, tid in enumerate(nuc_ids)}
nuc_labels = list("ACGT")

for ax, (name, model) in zip(axes, [("Caduceus-Ph", ph), ("Caduceus-PS", ps)]):
    cm = np.zeros((4, 4), dtype=np.int64)
    with torch.no_grad():
        logits = model(masked_ids).logits
    masked_logits = logits[mask]
    masked_true = ids_eval[mask]
    pred = masked_logits.argmax(dim=-1)
    for t, p in zip(masked_true.cpu().tolist(), pred.cpu().tolist()):
        if t in nuc_idx and p in nuc_idx:
            cm[nuc_idx[t], nuc_idx[p]] += 1
    # Row-normalize so each row sums to 1 (probability of predicting X | true is Y)
    row_sum = cm.sum(axis=1, keepdims=True).clip(min=1)
    cm_norm = cm / row_sum
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(4)); ax.set_xticklabels(nuc_labels)
    ax.set_yticks(range(4)); ax.set_yticklabels(nuc_labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("true (masked)")
    ax.set_title(f"{name}: per-base prediction\n(row-normalized)")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]*100:.0f}%",
                    ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black",
                    fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 7. Performance: wall-clock vs sequence length

Sequential scan in `Mamba._ssm` dominates; per-position cost is roughly constant in `L`. Linear fit of total time vs. `L` gives the effective μs/position. Extrapolating to `L=131,072` gives the full pretrained-context forward time.

In [ ]:
def bench_forward(model, L, n_runs=3, n_warmup=1):
    rand_ids = torch.randint(7, 11, (1, L), dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(rand_ids)
    if device.type == "mps":
        torch.mps.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(rand_ids)
            if device.type == "mps":
                torch.mps.synchronize()
    return (time.perf_counter() - t0) / n_runs

lengths = [128, 256, 512, 1024, 2048, 4096]
times = [bench_forward(ph, L) for L in lengths]
for L, t in zip(lengths, times):
    print(f"L={L:>5d}: {t:>6.3f} s  ({t / L * 1e6:>6.1f} μs/pos)")

# Linear extrapolation to full pretrained context
slope, intercept = np.polyfit(lengths, times, 1)
L_full = 131072
print(f"\nlinear fit: {slope*1e6:.2f} μs/pos + {intercept*1e3:.1f} ms fixed")
print(f"extrapolated L=131,072: {slope*L_full + intercept:.0f} s ≈ {(slope*L_full + intercept)/60:.1f} min")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(lengths, times, "o-")
ax1.plot(lengths, [slope*L + intercept for L in lengths], "--", alpha=0.5, label="linear fit")
ax1.set_xlabel("sequence length")
ax1.set_ylabel("forward time (s)")
ax1.set_title(f"Caduceus-Ph forward on {device}")
ax1.grid(True)
ax1.legend()

us_per_pos = [t / L * 1e6 for L, t in zip(lengths, times)]
ax2.plot(lengths, us_per_pos, "o-")
ax2.set_xlabel("sequence length")
ax2.set_ylabel("μs/position")
ax2.set_title("Per-position cost (should be ~flat)")
ax2.grid(True)
plt.tight_layout()
plt.show()

## 8. Memory profile

Per Mamba call, the largest live tensors are `deltaA` and `deltaB_u` of shape `(B, L, d_inner, n)`. Estimated peak working set as a function of `L`.

In [ ]:
d_inner = ph.caduceus.backbone.layers[0].mixer.mamba_fwd.d_inner  # 512
n_state = ph.caduceus.backbone.layers[0].mixer.mamba_fwd.A_log.shape[1]  # 16
d_model = ph.config.d_model  # 256

# Per-position memory cost (B=1, fp32):
#   deltaA + deltaB_u : 2 * d_inner * n_state * 4 bytes
#   Mamba transients  : ~5 * d_inner * 4   (xz, x, z, conv1d output, y)
#   Inter-layer state : ~3 * d_model * 4   (residual + hidden + 1 spare)
per_pos_bytes = 2 * d_inner * n_state * 4 + 5 * d_inner * 4 + 3 * d_model * 4
fixed_bytes = sum(p.numel() * 4 for p in {p.data_ptr(): p for p in ph.parameters()}.values()) + 50_000_000

lens = np.array([1024, 4096, 16384, 65536, 131072, 262144])
peak_gb = (per_pos_bytes * lens + fixed_bytes) / 1e9
for L, gb in zip(lens, peak_gb):
    print(f"L={L:>7,}: ~{gb:>5.2f} GB peak (B=1, fp32)")

plt.figure(figsize=(8, 4))
plt.plot(lens, peak_gb, "o-")
plt.axhline(8, color="orange", linestyle="--", label="8 GB Mac M-class budget (~)")
plt.axhline(16, color="red", linestyle="--", label="16 GB ceiling (~)")
plt.axhline(40, color="darkred", linestyle="--", label="48 GB realistic (~)")
plt.xlabel("sequence length")
plt.ylabel("estimated peak memory (GB)")
plt.title("Caduceus-Ph estimated memory vs L (B=1)")
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

## 9. Effective memory spectrum

For each `(channel, state-dim)` pair, the effective receptive field is `1 / (dt · |A|)`. We use `softplus(dt_proj.bias)` as the input-independent baseline (the actual `dt` is `softplus(W·x + bias)`, so this lower-bounds memory at typical inputs).

In [ ]:
def collect_effective_memories(model):
    layer_memories = []
    for layer in model.caduceus.backbone.layers:
        # rcps=False: layer.mixer is BiMambaWrapper. rcps=True: layer.mixer.submodule.
        mixer = layer.mixer
        if hasattr(mixer, "submodule"):
            mixer = mixer.submodule
        with torch.no_grad():
            mamba = mixer.mamba_fwd
            A = -torch.exp(mamba.A_log.float()).cpu()
            dt_baseline = F.softplus(mamba.dt_proj.bias.float()).cpu()
            mem = (1.0 / (dt_baseline.unsqueeze(-1) * A.abs())).flatten().numpy()
        layer_memories.append(mem)
    return layer_memories

ph_mems = collect_effective_memories(ph)
ps_mems = collect_effective_memories(ps)

ph_all = np.concatenate(ph_mems)
ps_all = np.concatenate(ps_mems)

for name, mem in [("Caduceus-Ph", ph_all), ("Caduceus-PS", ps_all)]:
    qs = np.quantile(mem, [0.1, 0.5, 0.9, 0.99])
    print(f"{name}: p10={qs[0]:6.1f} bp  p50={qs[1]:6.1f} bp  p90={qs[2]:7.1f} bp  p99={qs[3]:9.1f} bp  max={mem.max():.0f} bp")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, mem) in zip(axes, [("Caduceus-Ph", ph_all), ("Caduceus-PS", ps_all)]):
    ax.hist(np.log10(np.maximum(mem, 1)), bins=80)
    ax.set_xlabel("log10(effective memory in bp)")
    ax.set_title(f"{name}: {len(mem):,} (channel × state) pairs")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
# Per-layer median effective memory — does the model use shorter or longer scales at depth?
fig, ax = plt.subplots(figsize=(10, 4))
for name, mems in [("Caduceus-Ph", ph_mems), ("Caduceus-PS", ps_mems)]:
    medians = [np.median(m) for m in mems]
    p90s = [np.quantile(m, 0.9) for m in mems]
    ax.plot(medians, "o-", label=f"{name} median")
    ax.plot(p90s, "s--", alpha=0.6, label=f"{name} p90")
ax.set_xlabel("layer index")
ax.set_ylabel("effective memory (bp)")
ax.set_yscale("log")
ax.set_title("Per-layer effective memory across depth")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. dt and A_log structure

`dt` is the discretization step; `A` is the diagonal SSM matrix (always negative). The product `dt · |A|` controls per-step decay. Visualizing these shows whether the model uses a uniform dynamics regime or learns specialized channels.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# dt baseline distribution per layer
for layer_idx, layer in enumerate(ph.caduceus.backbone.layers):
    mixer = layer.mixer.submodule if hasattr(layer.mixer, "submodule") else layer.mixer
    with torch.no_grad():
        dt = F.softplus(mixer.mamba_fwd.dt_proj.bias.float()).cpu().numpy()
    bp = ax1.boxplot(dt, positions=[layer_idx], widths=0.6, showfliers=False, patch_artist=True)
    bp["boxes"][0].set_facecolor("steelblue")
    bp["boxes"][0].set_alpha(0.5)
ax1.set_xlabel("layer")
ax1.set_ylabel("softplus(dt_proj.bias)")
ax1.set_yscale("log")
ax1.set_title("Caduceus-Ph: dt baseline per layer")
ax1.grid(True, alpha=0.3)

# |A| distribution per layer
for layer_idx, layer in enumerate(ph.caduceus.backbone.layers):
    mixer = layer.mixer.submodule if hasattr(layer.mixer, "submodule") else layer.mixer
    with torch.no_grad():
        A = torch.exp(mixer.mamba_fwd.A_log.float()).cpu().numpy().flatten()
    bp = ax2.boxplot(A, positions=[layer_idx], widths=0.6, showfliers=False, patch_artist=True)
    bp["boxes"][0].set_facecolor("coral")
    bp["boxes"][0].set_alpha(0.5)
ax2.set_xlabel("layer")
ax2.set_ylabel("|A| = exp(A_log)")
ax2.set_yscale("log")
ax2.set_title("Caduceus-Ph: |A| per layer")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# A_log heatmap for a few layers — visualizes the (d_inner, d_state) structure of the diagonal SSM matrix
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, layer_idx in zip(axes, [0, 8, 15]):
    mixer = ph.caduceus.backbone.layers[layer_idx].mixer
    mixer = mixer.submodule if hasattr(mixer, "submodule") else mixer
    with torch.no_grad():
        A = -torch.exp(mixer.mamba_fwd.A_log.float()).cpu().numpy()
    im = ax.imshow(A, aspect="auto", cmap="RdBu")
    ax.set_xlabel("state dim (n=16)")
    ax.set_ylabel("channel (d_inner=512)")
    ax.set_title(f"Caduceus-Ph layer {layer_idx}: A matrix")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 11. Activation sparsity & dead channels

Two views:
1. **Sparsity in the residual stream** at each layer (fraction of activations near zero) — useful for understanding where information lives.
2. **Dead channels in `A_log`**: very-large `|A|` × any `dt` ⇒ instant decay ⇒ channel doesn't carry information across positions. These channels are effectively local-only.

In [ ]:
# Capture residual stream sparsity per layer using a forward hook on each Block
stream_stats = []
hooks = []

def make_hook(layer_idx):
    def hook(module, inputs, outputs):
        # outputs is (hidden_states, residual)
        h, r = outputs
        stream_stats.append({
            "layer": layer_idx,
            "h_abs_mean": h.abs().mean().item(),
            "h_near_zero_frac": (h.abs() < 1e-3).float().mean().item(),
            "r_abs_mean": r.abs().mean().item(),
        })
    return hook

for i, layer in enumerate(ph.caduceus.backbone.layers):
    hooks.append(layer.register_forward_hook(make_hook(i)))

with torch.no_grad():
    _ = ph(ids)

for h in hooks:
    h.remove()

print(f"{'layer':>5} {'h_abs_mean':>12} {'h_near_zero_frac':>18} {'r_abs_mean':>12}")
for s in stream_stats:
    print(f"{s['layer']:>5d} {s['h_abs_mean']:>12.4f} {s['h_near_zero_frac']:>18.4f} {s['r_abs_mean']:>12.4f}")

layers_x = [s["layer"] for s in stream_stats]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(layers_x, [s["h_near_zero_frac"] for s in stream_stats], "o-", label="hidden near-zero (|h| < 1e-3)")
ax.plot(layers_x, [s["h_abs_mean"] for s in stream_stats], "s--", label="hidden |mean|")
ax.plot(layers_x, [s["r_abs_mean"] for s in stream_stats], "^--", label="residual |mean|")
ax.set_xlabel("layer")
ax.set_title("Caduceus-Ph: residual stream activity per layer")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Dead channels: percent of (channel, state) pairs whose memory < 1 bp at typical dt baseline
for name, mems in [("Caduceus-Ph", ph_mems), ("Caduceus-PS", ps_mems)]:
    all_mem = np.concatenate(mems)
    dead_frac = (all_mem < 1.0).mean()
    very_long = (all_mem > 10000).mean()
    print(f"{name}: dead (<1 bp) = {dead_frac*100:5.1f}%   long-range (>10k bp) = {very_long*100:4.1f}%")

## 12. RC equivariance verification (Caduceus-PS)

Architectural guarantee: `softmax(forward(x))[t, v] == softmax(forward(rc(x)))[L-1-t, complement(v)]`. Anything above ~1e-5 indicates a broken port (or numerical issue stacking up across 16 layers — which we already verified isn't the case).

In [ ]:
cmap = {int(k): v for k, v in ps.config.complement_map.items()}
rc_ids = torch.tensor([[cmap[t] for t in ids[0].tolist()][::-1]], dtype=torch.long).to(device)

with torch.no_grad():
    sm = F.softmax(ps(ids).logits, dim=-1)[0]
    sm_rc = F.softmax(ps(rc_ids).logits, dim=-1)[0]

cmap_idx = torch.tensor([cmap[i] for i in range(ps.config.vocab_size)], device=device)
sm_rc_aligned = torch.flip(sm_rc[..., cmap_idx], dims=[0])
diff = (sm - sm_rc_aligned).abs()

print(f"max abs diff in softmax: {diff.max().item():.2e}")
print(f"mean abs diff in softmax: {diff.mean().item():.2e}")
print(f"(architectural property; should be ~1e-5 or smaller on real weights)")

## 13. Suggested follow-ups

Things this notebook deliberately doesn't cover but are likely interesting:

- **Real biological accuracy**: replace the synthetic `seq` in Section 6 with held-out chromosome chunks (e.g., the Caduceus paper's evaluation regions on hg38). The methodology is identical; only the data source changes.
- **Mechanistic probing**: feed canonical motifs (TATA box, Kozak, splice sites) and look at which channels light up — pair with `forward_hook` on `mamba_fwd` per layer.
- **Embedding geometry**: PCA on the 16 token embeddings to see how the model represents nucleotides vs. special tokens; expect A/T and C/G to be related by the complement involution in PS.
- **Long-context probe**: run on a real chromosome chunk at L=131k and compare last-position embedding vs. windowed inference — quantifies how much long-range integration the trained channels actually do.
- **Effective rank of `x_proj`/`out_proj`**: SVD of the per-layer projections — if many singular values are ~zero, parts of the model are unused capacity.
- **Compare with sequential vs parallel scan** once Step 9 (log-space Heinsen) lands: same logits within fp32 tolerance, but multi-x speedup expected.
- **Sparsification studies**: prune the ~24% dead `(channel, state)` pairs and measure quality drop — the dt/A spectrum suggests substantial unused state-space capacity.
- **VEP variant scoring** as a downstream eval: uses the existing `vep_embeddings.py` pipeline ported to MPS as the consumer of this model.